In [1]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [2]:
import sys

In [3]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.table import TableChunker
from financial_qa.chunkers.table_aware_recursive import TableAwareRecursiveChunker
from financial_qa.chunkers.synthetic_summary import SummaryChunker
from financial_qa.chunkers.table_split import TableSplitChunker
from financial_qa.chunkers.semantic import SemanticChunker
from financial_qa.chunkers.synthetic_table_row_to_text import TableRowToTextChunker
from financial_qa.preprocessors import RegulatoryReportPreprocessor
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.agent.gigachat_agent_loop import GigaChatAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [4]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
OMC_API_KEY = os.getenv('OMC_API_KEY')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = None

RAG_DB = 'giga_embeddings_summary_table_split_semantic'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
SYNTH_CHUNK_MODEL = "google/gemma-4-26b-a4b-it"
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

GEN_MODEL = "GigaChat-2-Pro"
QUERY_CONCURRENCY = 10

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
# JUDGE_MODEL = 'opus-4.7'
# JUDGE_MODEL = 'google/gemma-4-26b-a4b-it'
JUDGE_PROCESSES = 100  # None => one process per question
USE_GIGACHAT_JUDGE = True

In [5]:
all_records = load_jsonl(DATASET_FILE)

In [6]:
len(all_records)

449

In [7]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

449
Loaded 449 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [8]:
RAW_DATA_DIR = 'data/parsed'
DATA_DIR = 'data/preprocessed'

preprocessed_dir = Path(DATA_DIR)
has_preprocessed = preprocessed_dir.exists() and any(preprocessed_dir.rglob('*.md'))
if not has_preprocessed:
    print(f'Preprocessing {RAW_DATA_DIR} -> {DATA_DIR}...')
    preprocessor = RegulatoryReportPreprocessor()
    outputs = preprocessor.preprocess_dir(RAW_DATA_DIR, DATA_DIR)
    print(f'Preprocessed {len(outputs)} file(s).')
else:
    print(f'Using existing preprocessed data at {preprocessed_dir}')

Using existing preprocessed data at preprocessed_data


In [9]:
chunker = SummaryChunker(
    base_chunker=TableSplitChunker(
            text_chunker=SemanticChunker(
                chunk_size=CHUNK_SIZE,
                chunk_overlap=CHUNK_OVERLAP,
            ),
            max_rows_per_chunk=5,
        ),
    api_key=OPENROUTER_API_KEY,
    model=SYNTH_CHUNK_MODEL,
    window_size=20,
)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS
)

rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/preprocessed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc(max_workers=1)
else:
    print(f'Using existing index at {store_dir}')

loop = OpenRouterAgentLoop(
    rag=rag,
    max_turns=8,
    model="Anthropic/opus-4.7",
    base_url="https://api.ohmycode.ai/v1/",
    api_key=OMC_API_KEY
)

Using existing index at indexes/giga_embeddings_summary_table_split_semantic


In [10]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/449 [00:00<?, ?question/s]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 13s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent:   0%|          | 2/449 [00:13<45:22,  6.09s/question, avg_s=11.05, errors=0, last_conf=0.8]  

[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 16s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 13s…


Querying agent:   1%|          | 5/449 [00:25<27:06,  3.66s/question, avg_s=16.31, errors=0, last_conf=0.83]

...

[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 16s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…


Querying agent:  97%|█████████▋| 435/449 [17:00<00:23,  1.68s/question, avg_s=23.21, errors=32, last_conf=0.82]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…


Querying agent: 100%|██████████| 449/449 [17:52<00:00,  2.39s/question, avg_s=23.44, errors=33, last_conf=0.924]

Done: 449 answers, 33 errors


In [11]:
query_errors

[{'question_id': 'q_029f46c431cd202e',
  'question': 'Какова доля выпущенных гарантий (за вычетом оценочного резерва под кредитные убытки) в общей сумме обязательств кредитного характера за вычетом резерва у ЗАО «Альфа-Банк» по состоянию на 31 декабря 2025 года?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': "400, message='Bad Request', url='https://api.ohmycode.ai/v1/chat/completions'",
  'elapsed_s': 52.74124995805323},
 {'question_id': 'q_167e29be08e126ac',
  'question': 'Какую долю от резерва под ожидаемые кредитные убытки по кредитам и авансам клиентам Россельхозбанка на 31 декабря 2024 года составил эффект от изменений в методологии определения дефолта (актуализация критериев вынужденной реструктуризации и порогов материальности), описанных в учётной политике?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': "400, message='Bad Request', url='https://api.ohmycode.ai/v1/chat/completions'",
  'elapsed_s': 67.10695320786908},
 {'question_id':

In [12]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 449/449 [00:05<00:00, 74.87question/s, accuracy=83.96%, correct=377, errors=0] 


In [13]:
print(result["correct"])

377


In [14]:
accuracy = result['correct'] / result['total']

conf_scores = []

correct_conf = []
incorrect_conf = []

for res in result['results']:
    confidence = predicted.get(res['question_id'], {}).get('confidence')
    if confidence is None:
        continue
    score = confidence if res['judge_score'] == 1 else 1 - confidence
    conf_scores.append(score)

    if res['judge_score'] == 1:
        correct_conf.append(confidence)
    else:
        incorrect_conf.append(confidence)

conf_precision = sum(conf_scores) / len(conf_scores) if conf_scores else float('nan')

print(f"Accuracy:                   {accuracy:.2%} ({result['correct']}/{result['total']})")
print(f"Confidence precision:       {conf_precision:.4f}")
print(f"Avg confidence (correct):   {sum(correct_conf)/len(correct_conf):.4f}")
print(f"Avg confidence (incorrect): {sum(incorrect_conf)/len(incorrect_conf):.4f}")

Accuracy:                   83.96% (377/449)
Confidence precision:       0.7904
Avg confidence (correct):   0.8403
Avg confidence (incorrect): 0.6789


In [15]:
for res in result['results'][0:5]:
    print(res['question'])
    print(res['gold_answer'])
    print(res['predicted_answer'])
    print(res['judge_reasoning'])
    print(res['judge_score'])
    print(res['question_id'])
    print('-' * 75)

Каковы чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года?
5 592 млн руб.
Чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года, составили 5 592 млн руб. (комиссионные доходы 6 374 млн руб. за вычетом комиссионных расходов 782 млн руб.). Для сравнения, за аналогичный период 2024 года показатель составлял 1 742 млн руб.
Предсказанный ответ содержит правильное значение чистых комиссионных доходов, указанное в золотом ответе.
1
q_009c97884b8dc010
---------------------------------------------------------------------------
Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?
1,151 тыс. белорусских рублей
По состоянию на 31 декабря 2025 года общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, составила 1 151 тыс. белорусских рублей (в том числе 